# Read-Level Benchmarking: All Methods

Compares per-read modification probability scores across **all implemented baleen methods** and optionally against **external tools** (m6Anet, CHEUI, Dorado).

## Methods Compared (Baleen Internal)
| Label | Source | Description |
|-------|--------|-------------|
| `DTI` | `distance_to_ivt` | Log-median DTW distance to IVT pool, calibrated |
| `kNN` | `knn_ivt_purity` | kNN IVT-purity score, Beta EM calibrated |
| `MDS+GMM` | `mds_gmm` | MDS embedding + Gaussian mixture posterior |
| `V1 z→p` | hierarchical V1 | z-score converted to 1-sided tail probability |
| `V2 raw` | `p_mod_raw` | Anchored 2-component mixture EM with soft gating |
| `V2 kNN` | `p_mod_knn` | kNN IVT-purity (same as kNN, post-HMM input) |
| `V3 HMM` | `p_mod_hmm` | HMM forward-backward smoothed (from read_results.bam) |

## External Tools (Optional)
Set the file paths in **Section 2** if outputs are available.

## 1. Imports and Configuration

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, f1_score,
)
from sklearn.calibration import calibration_curve
from scipy import stats

# Baleen imports
from baleen.eventalign import load_results
from baleen.eventalign import (
    compute_sequential_modification_probabilities,
    load_read_results,
    distance_to_ivt,
    knn_ivt_purity,
    mds_gmm,
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('Imports OK')

## 2. File Paths — Edit These

In [ ]:
# ── Baleen outputs ───────────────────────────────────────────────────
BALEEN_PKL   = Path('../output/pipeline_results.pkl')   # from baleen run
BALEEN_BAM   = Path('../output/read_results.bam')       # per-read BAM (V3 HMM)

# ── External tool outputs (set to None to skip) ──────────────────────
# m6Anet: data.indiv_proba.csv  (columns: transcript_id, transcript_position,
#         read_index, probability_modified)
M6ANET_CSV   = None  # e.g. Path('../m6anet/data.indiv_proba.csv')

# CHEUI: per-read TSV  (columns: read_id, chromosome, position,
#         strand, kmer, prob_modification)
CHEUI_TSV    = None  # e.g. Path('../cheui/read_level_predictions.tsv')

# Dorado modBAM: indexed BAM with MM/ML tags for m6A modification
DORADO_BAM   = None  # e.g. Path('../dorado/calls.bam')

# ── Ground-truth position offset ─────────────────────────────────────
# Pipeline internally converts f5c 0-based first-of-kmer to 1-based center-of-kmer.
# Pipeline positions therefore match biological 1-based positions directly.
POSITION_OFFSET = 0

# ── Output directory for saved figures ───────────────────────────────
OUT_DIR = Path('figures_read_level')
OUT_DIR.mkdir(exist_ok=True)

print('Configuration set.')

## 3. Ground Truth: Known Modification Sites (E. coli rRNA)

In [ ]:
KNOWN_MODIFICATIONS = {
    # Pseudouridine (Ψ)
    ('ecoli16S', 516):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 746):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 955):  ('Ψ',     'pseudouridine'),
    ('ecoli23S', 1911): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 1917): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2457): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2504): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2580): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2604): ('Ψ',     'pseudouridine'),
    ('ecoli23S', 2605): ('Ψ',     'pseudouridine'),
    # N2-methylguanosine (m2G)
    ('ecoli16S', 966):  ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1207): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1516): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 1835): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 2445): ('m2G',   'N2-methylguanosine'),
    # 5-methylcytidine (m5C)
    ('ecoli16S', 967):  ('m5C',   '5-methylcytidine'),
    ('ecoli16S', 1407): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 1962): ('m5C',   '5-methylcytidine'),
    # 5-methyluridine (m5U / ribothymidine)
    ('ecoli23S', 747):  ('m5U',   '5-methyluridine'),
    ('ecoli23S', 1939): ('m5U',   '5-methyluridine'),
    # N6,N6-dimethyladenosine (m6,6A)
    ('ecoli16S', 1518): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli16S', 1519): ('m6,6A', 'N6,N6-dimethyladenosine'),
    # N6-methyladenosine (m6A)
    ('ecoli23S', 1618): ('m6A',   'N6-methyladenosine'),
    ('ecoli23S', 2030): ('m6A',   'N6-methyladenosine'),
    # N7-methylguanosine (m7G)
    ('ecoli16S', 527):  ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2069): ('m7G',   'N7-methylguanosine'),
    # Other modifications
    ('ecoli23S', 2498): ('Cm',    "2'-O-methylcytosine"),
    ('ecoli23S', 2449): ('D',     'dihydrouridine'),
    ('ecoli23S', 2251): ('Gm',    "2'-O-methylguanosine"),
    ('ecoli23S', 2552): ('Um',    "2'-O-methyluridine"),
    ('ecoli23S', 2501): ('ho5C',  '5-hydroxycytidine'),
    ('ecoli23S', 745):  ('m1G',   '1-methylguanosine'),
    ('ecoli23S', 2503): ('m2A',   '2-methyladenosine'),
    ('ecoli23S', 1915): ('m3Ψ',   '3-methylpseudouridine'),
    ('ecoli16S', 1498): ('m3U',   '3-methyluridine'),
    ('ecoli16S', 1402): ('m4Cm',  "N4,2'-O-dimethylcytidine"),
}

# Convert to 0-based pipeline positions
KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f'Total known modification sites: {len(KNOWN_MODIFICATIONS)}')
print(f'Modification types: {len(mod_counts)}')
print()
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f'  {mod_type:8s}  {count} sites   ({full})')

## 4. Load Baleen Pipeline Results

In [ ]:
contig_results = None

# Search default locations if configured path not found
for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        print(f'Loading: {candidate}')
        contig_results, metadata = load_results(candidate)
        break

if contig_results is None:
    raise FileNotFoundError(
        'pipeline_results.pkl not found. Run:\n'
        '  baleen run --native-bam ... --ivt-bam ... --ref ... -o output/'
    )

print(f'\nLoaded {len(contig_results)} contigs:')
for contig, cr in contig_results.items():
    n_pos = len(cr.positions)
    known_in_contig = sum(1 for p in cr.positions if (contig, p) in KNOWN_MOD_SET)
    print(f'  {contig}: {n_pos:,} positions ({known_in_contig} known mod sites)')

## 5. Run All Baleen Probability Methods

For each position, compute:
- **DTI**: `distance_to_ivt` (log-median distance, calibrated)
- **kNN**: `knn_ivt_purity` (kNN IVT-purity, Beta EM)
- **MDS+GMM**: `mds_gmm` (MDS + Gaussian mixture posterior)
- **V1→V3**: full hierarchical pipeline (z-scores, p_mod_raw, p_mod_knn, p_mod_hmm)

In [ ]:
print('Running hierarchical pipeline (V1 → V3)...')
hier_results = {}
for contig, cr in contig_results.items():
    hier_results[contig] = compute_sequential_modification_probabilities(cr)
    print(f'  {contig}: done')
print('Hierarchical pipeline complete.')

In [ ]:
def extract_read_scores(contig_results, hier_results, known_mod_set, known_mod_pipeline):
    """Extract per-read scores for all baleen methods."""
    records = []
    for contig, cr in contig_results.items():
        cmr = hier_results.get(contig)
        for pos, pr in cr.positions.items():
            is_mod = (contig, pos) in known_mod_set
            mod_info = known_mod_pipeline.get((contig, pos), ('unmod', 'unmodified'))
            n_nat = pr.n_native_reads
            n_ivt = pr.n_ivt_reads

            if n_nat + n_ivt < 4:
                continue

            # Run the three probability algorithms
            try:
                dti  = distance_to_ivt(pr.distance_matrix, n_nat, n_ivt)
                knn  = knn_ivt_purity(pr.distance_matrix, n_nat, n_ivt)
                mds  = mds_gmm(pr.distance_matrix, n_nat, n_ivt)
            except Exception:
                continue

            # Retrieve hierarchical outputs
            ps = cmr.position_stats.get(pos) if cmr else None

            # Build one record per read (native + IVT)
            all_names = list(pr.native_read_names) + list(pr.ivt_read_names)
            is_native_flags = [True] * n_nat + [False] * n_ivt

            for i, (name, is_nat) in enumerate(zip(all_names, is_native_flags)):
                y = 1 if (is_mod and is_nat) else 0  # modified = native @ known site

                rec = {
                    'contig':    contig,
                    'position':  pos,
                    'read_name': name,
                    'is_native': is_nat,
                    'y_true':    y,
                    'mod_type':  mod_info[0],
                    # raw probability methods
                    'dti':       float(dti.probabilities[i]),
                    'knn':       float(knn.probabilities[i]),
                    'mds_gmm':   float(mds.probabilities[i]),
                }

                if ps is not None:
                    # Convert z-score to 1-sided tail prob (higher = more modified)
                    z = float(ps.z_scores[i])
                    rec['v1_z2p']   = float(stats.norm.sf(-z))  # P(Z > -z)
                    rec['v2_raw']   = float(ps.p_mod_raw[i])
                    rec['v2_knn']   = float(ps.p_mod_knn[i])
                    rec['v3_hmm']   = float(ps.p_mod_hmm[i])
                else:
                    rec.update({'v1_z2p': np.nan, 'v2_raw': np.nan,
                                'v2_knn': np.nan, 'v3_hmm': np.nan})

                records.append(rec)

    return pd.DataFrame(records)


df_baleen = extract_read_scores(contig_results, hier_results, KNOWN_MOD_SET, KNOWN_MOD_PIPELINE)
print(f'Read records: {len(df_baleen):,}')
print(f'  Modified (native @ known site): {df_baleen.y_true.sum():,}')
print(f'  Unmodified:                     {(df_baleen.y_true == 0).sum():,}')
df_baleen.head(3)

## 6. Load V3 HMM from read_results.bam

`read_results.bam` stores the final `p_mod_hmm` per read via the `MP:f` tag. This cross-validates the in-memory extraction above.

In [ ]:
df_bam = None

for candidate in [BALEEN_BAM,
                  Path('../output/read_results.bam'),
                  Path('../results/read_results.bam')]:
    if candidate is not None and candidate.exists():
        print(f'Loading read_results.bam: {candidate}')
        df_bam = load_read_results(str(candidate))
        break

if df_bam is not None:
    print(f'BAM records: {len(df_bam):,}')
    print(df_bam.head(3))
else:
    print('read_results.bam not found — V3 HMM will be taken from in-memory results only.')

## 7. Load External Tool Outputs (Optional)

In [ ]:
# ── m6Anet ──────────────────────────────────────────────────────────
# Expected columns: transcript_id, transcript_position, read_index,
#                   probability_modified
# Generated by: m6anet-inference --input_dir ... --out_dir ...
#               (outputs data.indiv_proba.csv for per-read probabilities)

df_m6anet = None
if M6ANET_CSV is not None and Path(M6ANET_CSV).exists():
    df_m6anet = pd.read_csv(M6ANET_CSV)
    # Rename to standard columns
    rename = {
        'transcript_id':       'contig',
        'transcript_position': 'position',
        'probability_modified': 'm6anet',
    }
    df_m6anet = df_m6anet.rename(columns={k: v for k, v in rename.items() if k in df_m6anet.columns})
    df_m6anet['position'] = df_m6anet['position'].astype(int) - 1  # 1-based → 0-based
    print(f'm6Anet: {len(df_m6anet):,} per-read records')
else:
    print('m6Anet: skipped (M6ANET_CSV not set or file not found)')

In [ ]:
# ── CHEUI ────────────────────────────────────────────────────────────
# Expected columns: read_id, chromosome, position, strand, kmer,
#                   prob_modification
# Generated by: cheui predict -I ... -m ... -o ...

df_cheui = None
if CHEUI_TSV is not None and Path(CHEUI_TSV).exists():
    df_cheui = pd.read_csv(CHEUI_TSV, sep='\t')
    rename = {
        'chromosome':       'contig',
        'read_id':          'read_name',
        'prob_modification': 'cheui',
    }
    df_cheui = df_cheui.rename(columns={k: v for k, v in rename.items() if k in df_cheui.columns})
    df_cheui['position'] = df_cheui['position'].astype(int) - 1  # 1-based → 0-based
    print(f'CHEUI: {len(df_cheui):,} per-read records')
else:
    print('CHEUI: skipped (CHEUI_TSV not set or file not found)')

In [ ]:
# ── Dorado modBAM ────────────────────────────────────────────────────
# Dorado basecaller embeds modification probabilities in MM/ML tags.
# Requires: pysam to parse the modBAM.
# The ML tag stores per-base probability in 0-255 scale.

df_dorado = None
if DORADO_BAM is not None and Path(DORADO_BAM).exists():
    import pysam
    records = []
    with pysam.AlignmentFile(str(DORADO_BAM), 'rb') as bam:
        for read in bam.fetch():
            if read.is_unmapped:
                continue
            # Get base modification probabilities (m6A: 'A+a' or 'A+m')
            try:
                mods = read.modified_bases  # dict: (base, strand, mod_code) -> list
            except AttributeError:
                continue  # older pysam
            if not mods:
                continue
            # Extract m6A probabilities at each position
            for (query_base, _, mod_code), positions_probs in mods.items():
                for qpos, prob_255 in positions_probs:
                    ref_pos = read.query_to_reference(qpos, one_based=False)
                    if ref_pos is None:
                        continue
                    records.append({
                        'read_name': read.query_name,
                        'contig':    read.reference_name,
                        'position':  ref_pos,
                        'dorado':    prob_255 / 255.0,
                    })
    df_dorado = pd.DataFrame(records)
    print(f'Dorado: {len(df_dorado):,} per-read per-position records')
else:
    print('Dorado: skipped (DORADO_BAM not set or file not found)')

## 8. Merge All Scores

Build a unified per-read DataFrame with one column per method. Missing methods → NaN.

In [ ]:
df = df_baleen.copy()

# Annotate ground truth for BAM-loaded rows (for cross-validation)
if df_bam is not None:
    df_bam_merged = df_bam.merge(
        df[['contig', 'position', 'read_name', 'y_true', 'mod_type']].drop_duplicates(),
        on=['contig', 'position', 'read_name'], how='left'
    )
    # Merge v3_hmm_bam back as a separate column for cross-check
    bam_col = df_bam_merged[['contig', 'position', 'read_name', 'p_mod_hmm']].rename(
        columns={'p_mod_hmm': 'v3_hmm_bam'}
    )
    df = df.merge(bam_col, on=['contig', 'position', 'read_name'], how='left')

# Merge m6Anet (join on contig + position + read_name if available)
if df_m6anet is not None:
    key_cols = ['contig', 'position']
    if 'read_name' in df_m6anet.columns or 'read_index' in df_m6anet.columns:
        if 'read_index' in df_m6anet.columns:
            df_m6anet = df_m6anet.rename(columns={'read_index': 'read_name'})
        key_cols.append('read_name')
    m6a_col = df_m6anet[key_cols + ['m6anet']]
    df = df.merge(m6a_col, on=key_cols, how='left')

# Merge CHEUI
if df_cheui is not None:
    key_cols = ['contig', 'position']
    if 'read_name' in df_cheui.columns:
        key_cols.append('read_name')
    cheui_col = df_cheui[key_cols + ['cheui']]
    df = df.merge(cheui_col, on=key_cols, how='left')

# Merge Dorado
if df_dorado is not None:
    dorado_col = df_dorado[['contig', 'position', 'read_name', 'dorado']]
    df = df.merge(dorado_col, on=['contig', 'position', 'read_name'], how='left')

print(f'Unified DataFrame: {len(df):,} reads x {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
print(f'\nPositive (modified at known site): {df.y_true.sum():,}')
print(f'Negative: {(df.y_true == 0).sum():,}')

## 9. Define Methods Table

Centralized registry of all available methods, colors, and display names.

In [ ]:
# (column_name, display_label, color, group)
ALL_METHODS = [
    ('dti',        'DTI',           '#78909C', 'baleen-raw'),
    ('knn',        'kNN',           '#42A5F5', 'baleen-raw'),
    ('mds_gmm',    'MDS+GMM',       '#26C6DA', 'baleen-raw'),
    ('v1_z2p',     'V1 z→p',        '#AB47BC', 'baleen-v1'),
    ('v2_raw',     'V2 raw',        '#66BB6A', 'baleen-v2'),
    ('v2_knn',     'V2 kNN',        '#29B6F6', 'baleen-v2'),
    ('v3_hmm',     'V3 HMM',        '#EF5350', 'baleen-v3'),
    ('v3_hmm_bam', 'V3 HMM (BAM)',  '#FF7043', 'baleen-v3'),
    ('m6anet',     'm6Anet',        '#FFA726', 'external'),
    ('cheui',      'CHEUI',         '#8D6E63', 'external'),
    ('dorado',     'Dorado',        '#5C6BC0', 'external'),
]

# Keep only columns that exist and have non-NaN data
available_methods = [
    (col, label, color, group)
    for col, label, color, group in ALL_METHODS
    if col in df.columns and df[col].notna().sum() > 10
]

print(f'Available methods: {len(available_methods)}')
for col, label, color, group in available_methods:
    n_valid = df[col].notna().sum()
    print(f'  [{group:12s}] {label:15s}  valid: {n_valid:,}')

## 10. ROC and PR Curves — All Methods

In [ ]:
y_true = df['y_true'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax_roc, ax_pr = axes

roc_summary = []
for col, label, color, group in available_methods:
    scores = df[col].values
    valid  = ~np.isnan(scores)
    if len(np.unique(y_true[valid])) < 2:
        continue

    fpr, tpr, _ = roc_curve(y_true[valid], scores[valid])
    roc_auc     = auc(fpr, tpr)
    prec, rec, _= precision_recall_curve(y_true[valid], scores[valid])
    pr_auc      = average_precision_score(y_true[valid], scores[valid])

    ls = '--' if group == 'external' else '-'
    ax_roc.plot(fpr, tpr, ls, color=color, lw=2,
                label=f'{label} (AUC={roc_auc:.3f})')
    ax_pr.plot(rec, prec, ls, color=color, lw=2,
               label=f'{label} (AP={pr_auc:.3f})')

    roc_summary.append(dict(method=label, group=group,
                            roc_auc=roc_auc, pr_auc=pr_auc,
                            n_valid=valid.sum()))

# Chance line
ax_roc.plot([0,1],[0,1],'k--',lw=1,label='Chance')
baserate = y_true.mean()
ax_pr.axhline(baserate, color='k', ls='--', lw=1,
              label=f'Chance ({baserate:.3f})')

for ax, xlabel, ylabel, title in [
    (ax_roc, 'False Positive Rate', 'True Positive Rate', 'ROC Curves — Read Level'),
    (ax_pr,  'Recall',              'Precision',          'Precision-Recall Curves — Read Level'),
]:
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8, loc='best')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(OUT_DIR / 'roc_pr_all_methods.pdf', bbox_inches='tight')
plt.show()

summary_df = pd.DataFrame(roc_summary).sort_values('roc_auc', ascending=False)
print(summary_df.round(4).to_string(index=False))

## 11. AUC Bar Chart — Grouped by Method Type

In [ ]:
if len(roc_summary) == 0:
    print('No summary data — run Section 10 first.')
else:
    sdf = pd.DataFrame(roc_summary).sort_values('roc_auc', ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, max(4, len(sdf) * 0.45)))
    group_colors = {
        'baleen-raw': '#42A5F5',
        'baleen-v1':  '#AB47BC',
        'baleen-v2':  '#66BB6A',
        'baleen-v3':  '#EF5350',
        'external':   '#FFA726',
    }

    for ax, metric, title in [(axes[0], 'roc_auc', 'ROC-AUC'),
                               (axes[1], 'pr_auc',  'PR-AUC (Average Precision)')]:
        bar_colors = [group_colors.get(g, 'gray') for g in sdf['group']]
        bars = ax.barh(sdf['method'], sdf[metric], color=bar_colors, edgecolor='white')
        ax.axvline(0.5 if metric == 'roc_auc' else y_true.mean(),
                   color='gray', ls='--', lw=1)
        ax.set_xlim([0, 1.05])
        ax.set_xlabel(title)
        ax.set_title(title)
        for bar, val in zip(bars, sdf[metric]):
            ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                    f'{val:.3f}', va='center', fontsize=9)

    # Legend for groups
    from matplotlib.patches import Patch
    legend_handles = [Patch(facecolor=c, label=g) for g, c in group_colors.items()
                      if g in sdf['group'].values]
    axes[0].legend(handles=legend_handles, title='Group', fontsize=8, loc='lower right')

    plt.suptitle('Read-Level AUC Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'auc_bar_all_methods.pdf', bbox_inches='tight')
    plt.show()

## 12. Score Distributions: Modified vs Unmodified

In [ ]:
n_methods = len(available_methods)
ncols = 4
nrows = (n_methods + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3))
axes = axes.flatten() if n_methods > 1 else [axes]

for ax, (col, label, color, group) in zip(axes, available_methods):
    scores = df[col].values
    valid  = ~np.isnan(scores)
    mod_s   = scores[valid & (y_true == 1)]
    unmod_s = scores[valid & (y_true == 0)]

    ax.hist(unmod_s, bins=30, alpha=0.55, density=True,
            color='#90A4AE', label=f'Unmod (n={len(unmod_s):,})')
    ax.hist(mod_s,   bins=30, alpha=0.70, density=True,
            color=color,     label=f'Mod   (n={len(mod_s):,})')

    # Overlap statistic (Bhattacharyya coefficient approximation via histogram)
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('P(modified)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.set_xlim([-0.05, 1.05])

# Hide unused subplots
for ax in axes[n_methods:]:
    ax.set_visible(False)

plt.suptitle('Score Distributions: Modified vs Unmodified Reads', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'score_distributions.pdf', bbox_inches='tight')
plt.show()

## 13. Calibration Curves

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3))
axes = axes.flatten() if n_methods > 1 else [axes]

for ax, (col, label, color, group) in zip(axes, available_methods):
    scores = df[col].values
    valid  = ~np.isnan(scores)
    if valid.sum() < 30 or len(np.unique(y_true[valid])) < 2:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.set_title(label)
        continue

    try:
        prob_true, prob_pred = calibration_curve(
            y_true[valid], scores[valid], n_bins=10, strategy='quantile'
        )
        ax.plot(prob_pred, prob_true, 'o-', color=color, lw=2, ms=5)
    except Exception as e:
        ax.text(0.5, 0.5, str(e)[:40], ha='center', va='center',
                transform=ax.transAxes, fontsize=7, color='red')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
    ax.set_title(f'{label}', fontweight='bold')
    ax.set_xlabel('Mean predicted')
    ax.set_ylabel('Fraction positive')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

for ax in axes[n_methods:]:
    ax.set_visible(False)

plt.suptitle('Calibration Curves (Read Level)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'calibration_read_level.pdf', bbox_inches='tight')
plt.show()

## 14. Performance by Modification Type

In [ ]:
# For each known modification type, compute one-vs-rest AUC across methods
mod_types = [mt for mt in df['mod_type'].unique() if mt != 'unmod']

mod_auc_records = []
for mt in mod_types:
    y_ovr = (df['mod_type'] == mt).astype(int).values  # all reads at that mod type as positive
    if y_ovr.sum() == 0:
        continue
    for col, label, color, group in available_methods:
        scores = df[col].values
        valid  = ~np.isnan(scores)
        if len(np.unique(y_ovr[valid])) < 2:
            continue
        try:
            roc_a = auc(*roc_curve(y_ovr[valid], scores[valid])[:2])
            pr_a  = average_precision_score(y_ovr[valid], scores[valid])
        except Exception:
            roc_a, pr_a = np.nan, np.nan
        mod_auc_records.append(dict(mod_type=mt, method=label, group=group,
                                    roc_auc=roc_a, pr_auc=pr_a,
                                    n_mod_reads=y_ovr.sum()))

mod_auc_df = pd.DataFrame(mod_auc_records)

if len(mod_auc_df) > 0:
    # Heatmap: modification types × methods
    pivot = mod_auc_df.pivot(index='mod_type', columns='method', values='roc_auc')
    # Sort rows by mean AUC descending
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns) * 1.2),
                                     max(6, len(pivot) * 0.5)))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=0.4, vmax=1.0, linewidths=0.5,
                ax=ax, cbar_kws={'label': 'ROC-AUC'})
    ax.set_title('Read-Level ROC-AUC by Modification Type (one-vs-rest)', fontsize=12)
    ax.set_ylabel('Modification Type')
    ax.set_xlabel('')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'auc_heatmap_by_mod_type.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Not enough per-modification-type data.')

## 15. Coverage-Stratified Performance

Performance can vary by read depth. Compare AUC across low / medium / high coverage positions.

In [ ]:
# Compute per-position coverage and bin into quartiles
pos_coverage = (
    df.groupby(['contig', 'position'])['is_native']
    .count()
    .reset_index(name='n_reads')
)
pos_coverage['depth_bin'] = pd.qcut(
    pos_coverage['n_reads'], q=3,
    labels=['Low (Q1)', 'Medium (Q2)', 'High (Q3)'],
    duplicates='drop'
)
df = df.merge(pos_coverage[['contig','position','depth_bin']],
              on=['contig','position'], how='left')

cov_records = []
for bin_label in df['depth_bin'].dropna().unique():
    mask = df['depth_bin'] == bin_label
    y_bin = y_true[mask]
    if len(np.unique(y_bin)) < 2:
        continue
    for col, label, color, group in available_methods:
        scores = df[col].values[mask]
        valid  = ~np.isnan(scores)
        if len(np.unique(y_bin[valid])) < 2:
            continue
        try:
            roc_a = auc(*roc_curve(y_bin[valid], scores[valid])[:2])
        except Exception:
            roc_a = np.nan
        cov_records.append(dict(depth_bin=bin_label, method=label,
                                group=group, roc_auc=roc_a,
                                n=mask.sum()))

if cov_records:
    cov_df = pd.DataFrame(cov_records)
    g = sns.catplot(
        data=cov_df, kind='bar',
        x='method', y='roc_auc', hue='depth_bin',
        height=5, aspect=2.2, palette='Set2'
    )
    g.set_xticklabels(rotation=35, ha='right')
    g.set_axis_labels('Method', 'ROC-AUC')
    g.ax.axhline(0.5, color='gray', ls='--', lw=1)
    g.ax.set_ylim([0, 1.05])
    g.fig.suptitle('Read-Level ROC-AUC by Read Depth', y=1.02)
    plt.savefig(OUT_DIR / 'auc_by_coverage.pdf', bbox_inches='tight')
    plt.show()

## 16. Precision-Recall at Fixed Thresholds

Precision and recall at P(mod) = 0.5 and at FDR-controlled thresholds.

In [ ]:
thresh_records = []
for threshold in [0.3, 0.5, 0.7]:
    for col, label, color, group in available_methods:
        scores = df[col].values
        valid  = ~np.isnan(scores)
        if valid.sum() == 0:
            continue
        yt = y_true[valid]
        yhat = (scores[valid] >= threshold).astype(int)
        tp = ((yhat == 1) & (yt == 1)).sum()
        fp = ((yhat == 1) & (yt == 0)).sum()
        fn = ((yhat == 0) & (yt == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        thresh_records.append(dict(
            method=label, group=group, threshold=threshold,
            precision=prec, recall=rec, f1=f1,
        ))

if thresh_records:
    tdf = pd.DataFrame(thresh_records)
    for t in [0.3, 0.5, 0.7]:
        subset = tdf[tdf['threshold'] == t].sort_values('f1', ascending=False)
        print(f'\n── Threshold = {t} ──────────────────────────────────')
        print(subset[['method','group','precision','recall','f1']].round(3).to_string(index=False))

## 17. V3 HMM Cross-Check: In-Memory vs BAM File

In [ ]:
if 'v3_hmm_bam' in df.columns and df['v3_hmm_bam'].notna().sum() > 0:
    common = df[['v3_hmm', 'v3_hmm_bam']].dropna()
    corr = common.corr().iloc[0, 1]
    print(f'V3 HMM in-memory vs BAM:  Pearson r = {corr:.5f}  (n={len(common):,})')

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.hexbin(common['v3_hmm'], common['v3_hmm_bam'],
              gridsize=50, cmap='Blues', mincnt=1)
    ax.set_xlabel('V3 HMM (in-memory)')
    ax.set_ylabel('V3 HMM (from BAM)')
    ax.set_title(f'Pearson r = {corr:.4f}')
    ax.plot([0,1],[0,1],'r--',lw=1)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'v3_hmm_crosscheck.pdf', bbox_inches='tight')
    plt.show()
else:
    print('BAM-loaded V3 HMM not available — skipping cross-check.')

## 18. Summary Table

In [ ]:
summary_records = []
for col, label, color, group in available_methods:
    scores = df[col].values
    valid  = ~np.isnan(scores)
    if len(np.unique(y_true[valid])) < 2:
        continue

    fpr, tpr, thresholds = roc_curve(y_true[valid], scores[valid])
    roc_a = auc(fpr, tpr)
    pr_a  = average_precision_score(y_true[valid], scores[valid])

    # Best F1 across all thresholds
    prec_, rec_, t_ = precision_recall_curve(y_true[valid], scores[valid])
    f1_ = 2 * prec_ * rec_ / np.maximum(prec_ + rec_, 1e-9)
    best_f1 = f1_.max()
    best_t  = t_[f1_[:-1].argmax()] if len(t_) > 0 else np.nan

    summary_records.append(dict(
        Group=group, Method=label,
        ROC_AUC=roc_a, PR_AUC=pr_a,
        Best_F1=best_f1, Best_Threshold=best_t,
        N_valid=int(valid.sum()),
    ))

sum_df = pd.DataFrame(summary_records).sort_values(['Group','ROC_AUC'], ascending=[True, False])

print('\n' + '='*75)
print('READ-LEVEL BENCHMARK SUMMARY')
print('='*75)
print(f'Reads: {len(df):,}  |  Positives: {y_true.sum():,}  |  Negatives: {(y_true==0).sum():,}')
print(f'Contigs: {df.contig.nunique()}  |  Positions: {df.groupby(["contig","position"]).ngroups:,}')
print()
print(sum_df.round(4).to_string(index=False))

sum_df.to_csv(OUT_DIR / 'read_level_summary.csv', index=False)
print(f'\nSaved to {OUT_DIR}/read_level_summary.csv')